In [1]:
import csv, json, os, shutil
from collections import Counter

# ---- STUDENT-SPECIFIC VALUES ----
ID_SUM = 27        # <- your ID_SUM
SEED_DIGIT = 6    # <- your SEED_DIGIT

N_RECORDS = 1000 + ID_SUM
TARGET_CLASS = (SEED_DIGIT % 4) + 2          # 2, 3, 4, or 5
SRC = r"C:\Users\Nate\Lab6_File_Handling_Linsangan\Lab6_File_handling_Additional_Exercises\archive (1)\weblog.csv"
OUT_DIR = "output"
BACKUP_DIR = "backup"
FILTERED = os.path.join(OUT_DIR, "filtered_records.csv")
SUMMARY = os.path.join(OUT_DIR, "summary.json")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BACKUP_DIR, exist_ok=True)

processed = invalid = 0
selected = []
status_counts = Counter()
url_counts = Counter()

# Req 1: stream line by line (csv.DictReader iterates lazily)
with open(SRC, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    print("Columns:", reader.fieldnames)   # confirm names match below
    for row in reader:
        if processed >= N_RECORDS:
            break
        processed += 1
        # Req 2: extract fields (adjust keys to the dataset's actual headers)
        ip   = (row.get("IP") or "").strip()
        time = (row.get("Time") or "").strip()
        url  = (row.get("URL") or "").strip()
        stat = (row.get("Staus") or row.get("Status") or "").strip()
        # Req 6: handle invalid/missing records
        if not (ip and time and url and stat.isdigit() and len(stat) == 3):
            invalid += 1
            continue
        # Req 3: keep only target class
        if int(stat) // 100 == TARGET_CLASS:
            selected.append((ip, time, url, stat))
            status_counts[stat] += 1      # Req 4: per status code
            url_counts[url] += 1

top5 = url_counts.most_common(5)

# Req 5: export filtered CSV + JSON summary
with open(FILTERED, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["IP", "Time", "URL", "Status"])
    w.writerows(selected)

summary = {
    "records_processed": processed,
    "invalid_or_missing_records": invalid,
    "target_status_class": f"{TARGET_CLASS}xx",
    "records_in_target_class": len(selected),
    "count_per_status_code": dict(status_counts),
    "top_5_urls": [{"url": u, "count": c} for u, c in top5],
    "filtered_csv_file": FILTERED,
}
with open(SUMMARY, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)

# Req 6: backup + verify
backup_path = os.path.join(BACKUP_DIR, "summary_backup.json")
shutil.copy2(SUMMARY, backup_path)
print("Backup exists:", os.path.exists(backup_path))
# Req 6: backup + verify
import hashlib

backup_path = os.path.join(BACKUP_DIR, "summary_backup.json")
shutil.copy2(SUMMARY, backup_path)

def sha256(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

exists = os.path.exists(backup_path)
size_match = exists and os.path.getsize(backup_path) == os.path.getsize(SUMMARY)
content_match = exists and sha256(backup_path) == sha256(SUMMARY)
print("BACKUP VERIFIED" if (exists and size_match and content_match) else "BACKUP FAILED")

# Assessment table values
print("Records processed:", processed)
print("Target status class:", f"{TARGET_CLASS}xx")
print("Records in target class:", len(selected))
print("Count per status code:", dict(status_counts))
print("Five most requested URLs:", top5)
print("Filtered CSV:", FILTERED)
print("JSON summary:", SUMMARY)
print("Invalid/missing records:", invalid)

Columns: ['IP', 'Time', 'URL', 'Staus']
Backup exists: True
BACKUP VERIFIED
Records processed: 1027
Target status class: 4xx
Records in target class: 5
Count per status code: {'404': 5}
Five most requested URLs: [('GET /robots.txt HTTP/1.1', 4), ('GET /favicon.ico HTTP/1.1', 1)]
Filtered CSV: output\filtered_records.csv
JSON summary: output\summary.json
Invalid/missing records: 0
